In [1]:
import os
import gc
import json
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from sklearn.metrics import (confusion_matrix, classification_report, ConfusionMatrixDisplay, fbeta_score,
                             recall_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support)
import matplotlib.pyplot as plt
import seaborn as sns
from torch.optim import Adam 
import torch.nn.functional as F
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.optim.lr_scheduler")

device = torch.device("cuda" if torch.cuda.is_available() else 
                     ("mps" if torch.backends.mps.is_available() else "cpu"))

def clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.append(str(ROOT))

from Utils.project_utils import *
import Utils.constants1 as c
from Utils.transforms import *
from Utils.MushroomDataset import *
from Utils.dataLoaders import *

In [3]:
print(f"Using device: {device}")

Using device: mps


In [4]:
SPLIT_DIR = Path.cwd().resolve() / "Data" / "splits"

In [5]:
# Load the datasets
train = pd.read_csv(SPLIT_DIR / "train.csv")
val = pd.read_csv(SPLIT_DIR / "val.csv")
test = pd.read_csv(SPLIT_DIR / "test.csv")

In [6]:
# Load the class mapping
with open(SPLIT_DIR / "class_to_idx.json", "r") as f:
    train_class_to_idx = json.load(f)

In [7]:
# Define transforms (fixing the syntax error from before)
train_tfms_mild = get_train_tfms_mild() 
train_tfms_strong = get_train_tfms_strong() 
val_tfms = get_val_tfms()

print("All files loaded successfully!")

All files loaded successfully!


In [8]:
save_dir = PROJECT_ROOT / "Graphs"
save_dir.mkdir(parents=True, exist_ok=True)

In [9]:
CLASSES = sorted(train["class"].unique().tolist())

In [10]:
loaders = make_loaders(
    train,
    val,
    test,
    train_tfms_mild,
    train_tfms_strong,
    val_tfms,
    train_class_to_idx,
    CLASSES,
    c.BATCH_SIZE,
    c.NUM_WORKERS,
    device,
)

loaders

{'train_loader_mild': <torch.utils.data.dataloader.DataLoader at 0x3a2a56cd0>,
 'train_loader_strong': <torch.utils.data.dataloader.DataLoader at 0x3a7ad67d0>,
 'val_loader': <torch.utils.data.dataloader.DataLoader at 0x3a7ad4310>,
 'test_loader': <torch.utils.data.dataloader.DataLoader at 0x3a7ae4810>}

In [11]:
train_loader = loaders["train_loader_strong"]
val_loader = loaders["val_loader"]
test_loader = loaders["test_loader"]

In [12]:
toxic_idx = train_class_to_idx['toxic']

In [13]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight,label_smoothing=0.1)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss)
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [14]:
# Calculate balanced class weights based on inverse frequency
class_counts = train['class'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(class_counts)  # Normalize
class_weights = class_weights.to(device)

print(f"Class counts: {class_counts}")
print(f"Class weights: {class_weights}")

criterion = FocalLoss(weight=class_weights, gamma=2.0)

Class counts: [12847 12846 12846]
Class weights: tensor([0.9999, 1.0000, 1.0000], device='mps:0')


## Training and Validation

### Load RESNET50 Model

In [15]:
clear_memory()

In [16]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01, betas=(0.9, 0.999))

In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

In [19]:
# warmup for 5 epochs, then Cosine Decay
num_epochs = 50

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)

In [20]:
main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - 5, eta_min=1e-6)

In [21]:
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[5])

In [22]:
BEST_MODEL_PATH = Path.cwd() / "best_models" / "resnet50_robustv3_best.pth"
BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Model initialized successfully!")

Model initialized successfully!


In [23]:
def run_one_epoch(model, loader, criterion, optimizer, device, toxic_idx, is_train=False, return_preds=False):
    running_loss, running_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    
    model.train() if is_train else model.eval()
    num_batches = len(loader)
    
    for i, (img, labels) in enumerate(loader):
        img, labels = img.to(device), labels.to(device)

        with torch.set_grad_enabled(is_train):
            outputs = model(img)
            loss = criterion(outputs, labels)
            
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(1)
        running_loss += loss.item() * img.size(0)
        running_correct += (preds == labels).sum().item()
        total += img.size(0)

        if (i + 1) % 5 == 0 or (i + 1) == num_batches:
            print(f"  {'Train' if is_train else 'Val'} [{i+1}/{num_batches}] Loss: {loss.item():.4f}", end='\r')

        if return_preds:
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    print() 
    avg_loss = running_loss / total
    avg_acc = running_correct / total

    if return_preds:
        y_true = torch.cat(all_labels).numpy()
        y_pred = torch.cat(all_preds).numpy()
        
        # Metrics Calculation
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1_m = f1_score(y_true, y_pred, average='macro')
        f1_w = f1_score(y_true, y_pred, average='weighted')
        
        # F2 - using toxic_idx here!
        prec, rec, f2_score, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[toxic_idx], beta=2.0, zero_division=0
        )
        _, _, f1_tox, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[toxic_idx], beta=1.0, zero_division=0
        )
        
        metrics = (avg_loss, avg_acc, bal_acc, f1_m, f1_w, prec[0], rec[0], f1_tox[0], f2_score[0])
        return metrics, y_true, y_pred
    
    return (avg_loss, avg_acc, None, None, None, None, None, None, None), None, None

In [24]:
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device, toxic_idx, patience=7, scheduler=None):
    best_macro_f1 = -float("inf")
    bad_epochs = 0

    for epoch in range(num_epochs):
        # training dataset 
        t_metrics, _, _ = run_one_epoch(
            model, train_loader, criterion, optimizer, device, toxic_idx, is_train=True  
        )
        t_loss, t_acc = t_metrics[0], t_metrics[1]
        
        # validation dataset 
        v_metrics, _, _ = run_one_epoch(
            model, val_loader, criterion, optimizer, device, toxic_idx, is_train=False, return_preds=True  
        )
        v_loss, v_acc, v_bal, v_f1_m, v_f1_w, v_tox_p, v_tox_r, v_tox_f1, v_prio = v_metrics

        current_lr = optimizer.param_groups[-1]['lr']
        
        if scheduler is not None:
            scheduler.step()

        print(f"\n{'='*20} EPOCH {epoch+1} {'='*20}")
        print(f"TRAIN: Loss {t_loss:.4f} | Acc {t_acc:.4f}")
        print(f"VAL:   Loss {v_loss:.4f} | Acc {v_acc:.4f} | Balanced: {v_bal:.4f}")
        print(f"TOXIC: Recall: {v_tox_r:.4f} | F1: {v_tox_f1:.4f} | F2: {v_prio:.4f}")
        print(f"MACRO F1: {v_f1_m:.4f} | Weighted F1: {v_f1_w:.4f}")
        print(f"LR:    {current_lr:.8f}")

        # early stopping 
        if v_f1_m > best_macro_f1:
            best_macro_f1 = v_f1_m
            bad_epochs = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f">>> New Best Model Saved! (Macro F1: {v_f1_m:.4f})")
        else:
            bad_epochs += 1
            print(f"No improvement. Patience: {bad_epochs}/{patience}")
            if bad_epochs >= patience:
                print(f"!!! Early stopping triggered.")
                break
        
        clear_memory()

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    return model

In [ ]:
model = train(
    model, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer, 
    num_epochs=50, 
    device=device, 
    toxic_idx=toxic_idx,  # Added this!
    patience=8, 
    scheduler=scheduler
)

  Train [20/151] Loss: 0.3566

In [ ]:
models_dir = PROJECT_ROOT / "best_models"
models_dir.mkdir(parents=True, exist_ok=True)
model_save_path = models_dir / "resnet50_Robustv3_Final.pth"
torch.save(model.state_dict(), model_save_path)

In [ ]:
print("FINAL EVALUATION ON TEST SET (standard prediction)")
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for img, labels in test_loader:
        img = img.to(device)
        outputs = model(img)
        preds = outputs.argmax(dim=1) 
        
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

y_true = torch.cat(all_labels).numpy()
y_pred = torch.cat(all_preds).numpy()

In [ ]:
accuracy = np.mean(y_true == y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1_m = f1_score(y_true, y_pred, average='macro')
f1_w = f1_score(y_true, y_pred, average='weighted')
prec, rec, f1_tox, _ = precision_recall_fscore_support(y_true, y_pred, labels=[toxic_idx], beta=1.0, zero_division=0)
_, _, f2_tox, _ = precision_recall_fscore_support(y_true, y_pred, labels=[toxic_idx], beta=2.0, zero_division=0)

In [ ]:
print(f"\nOverall Accuracy: {accuracy:.4f} (Balanced: {bal_acc:.4f})")
print(f"Macro F1: {f1_m:.4f} | Weighted F1: {f1_w:.4f}")
print(f"Toxic - Precision: {prec[0]:.4f}, Recall: {rec[0]:.4f}, F1: {f1_tox[0]:.4f}, F2: {f2_tox[0]:.4f}")
print("-" * 60)
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", values_format='d')
plt.title("Confusion Matrix: ResNet-50 Mushroom Classification", fontsize=14)
plt.tight_layout()
plt.savefig(save_dir / "resnet50_robust3_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
clear_memory()